IMPORT AND LOAD DATA 

In [29]:
import sys
sys.path.append("../src")

import pandas as pd

from config import get_country_settings, get_exog_specs
from data_utils import split_data
from sarimax_utils import sarimax_grid_search, compare_sarimax_models

In [30]:
df = pd.read_csv("../data/processed_data.csv", parse_dates=True, index_col=0)

df = df.sort_index()
df = df.asfreq("D")
df = df.dropna()

df.head()

,Total Load [MW] - Norway,Total Load [MW] - Sweden,Total Load [MW] - Finland,Temp - Norway,Temp - Sweden,Temp - Finland,WindSpeed - Norway,WindSpeed - Sweden,WindSpeed - Finland,Day of Week,...,month_March,month_April,month_May,month_June,month_July,month_August,month_September,month_October,month_November,month_December
Load Date,,,,,,,,,,,,,,,,,,,,,
2010-01-08,534842,577043,321334,-18.97,-13.95,-17.65,2.45,2.79,4.81,Friday,...,0,0,0,0,0,0,0,0,0,0
2010-01-09,519060,549194,309894,-16.23,-11.97,-15.30,2.53,2.50,2.67,Saturday,...,0,0,0,0,0,0,0,0,0,0
2010-01-10,497315,520101,289806,-12.60,-7.32,-10.72,2.57,2.08,1.62,Sunday,...,0,0,0,0,0,0,0,0,0,0
2010-01-11,519035,549179,295572,-9.51,-7.54,-6.13,2.47,1.68,3.19,Monday,...,0,0,0,0,0,0,0,0,0,0
2010-01-12,501649,552207,288217,-7.76,-5.95,-4.06,1.97,1.37,3.43,Tuesday,...,0,0,0,0,0,0,0,0,0,0


YEAR SETTINGS AND SPLIT, THEN SET THE COUNTRY 

In [31]:
train_start_year = 2010
forecast_year = 2025

val_year = forecast_year - 1
train_end_year = forecast_year - 2

train_data, val_data, development_data, forecast_data = split_data(
    df,
    train_start_year=train_start_year,
    forecast_year=forecast_year
)

print("=" * 70)
print(f"Train data       : {train_start_year}-01-01 to {train_end_year}-12-31")
print(f"Validation data  : {val_year}-01-01 to {val_year}-12-31")
print(f"Development data : {train_start_year}-01-01 to {val_year}-12-31")
print(f"Forecast data    : {forecast_year}-01-01 to {forecast_year}-12-31")
print("=" * 70)

Train data       : 2010-01-01 to 2023-12-31
Validation data  : 2024-01-01 to 2024-12-31
Development data : 2010-01-01 to 2024-12-31
Forecast data    : 2025-01-01 to 2025-12-31


In [32]:
country = "Norway"   # "Norway", "Sweden", "Finland"

code, target_col, lag_prefix = get_country_settings(country)
specs = get_exog_specs(country)

full_exog_vars = specs["Full"]
restricted_exog_vars = specs["Restricted"]

print("=" * 70)
print(f"Country: {country}")
print(f"Target : {target_col}")
print("=" * 70)

print("Exogenous variables by specification:")
for spec_name, exog_vars in specs.items():
    print(f"\n{spec_name}:")
    for var in exog_vars:
        print(f" - {var}")
print("=" * 70)

Country: Norway
Target : log_load_Norway
Exogenous variables by specification:

Full:
 - HDD_Norway
 - Extreme_Cold_NO
 - HDD_Extreme_NO
 - CDD_Norway
 - Extreme_Warm_NO
 - CDD_Extreme_NO
 - lag1_log_no
 - Holiday_Norway

Restricted:
 - HDD_Norway
 - Extreme_Cold_NO
 - HDD_Extreme_NO
 - lag1_log_no
 - Holiday_Norway


GRID SEARCH TO CHOOSE BEST SARIMAX MODEL

In [ ]:
grid_results, selected_summary_df, selected_models, final_results_dict = sarimax_grid_search(
    train_data=train_data,
    val_data=val_data,
    development_data=development_data,
    specs=specs,
    country=country,
    target_col=target_col,
    forecast_year=forecast_year,
    val_year=val_year,
    train_start_year=train_start_year,
    train_end_year=train_end_year,
    verbose=True
)

selected_summary_df

INSERT FINAL CHOSEN MODEL AND RUN

In [33]:
model_specs = {
    "SARIMAX Full": {
        "Specification": "Full",
        "exog_vars": full_exog_vars,
        "order": (1, 0, 0),
        "seasonal_order": (0, 1, 1, 7)
    },
    "SARIMAX Restricted": {
        "Specification": "Restricted",
        "exog_vars": restricted_exog_vars,
        "order": (0, 0, 1),
        "seasonal_order": (0, 1, 1, 7)
    }
}

print("=" * 70)
print(f"Country: {country}")
print(f"Target: {target_col}")
print("Model specifications:")

for model_name, spec in model_specs.items():
    print(
        f" - {model_name}: order={spec['order']}, "
        f"seasonal_order={spec['seasonal_order']}"
    )

print("=" * 70)

Country: Norway
Target: log_load_Norway
Model specifications:
 - SARIMAX Full: order=(1, 0, 0), seasonal_order=(0, 1, 1, 7)
 - SARIMAX Restricted: order=(0, 0, 1), seasonal_order=(0, 1, 1, 7)


Final estimation and comparison

In [34]:
sarimax_comparison_df, fitted_sarimax_models = compare_sarimax_models(
    model_specs=model_specs,
    development_data=development_data,
    forecast_data=forecast_data,
    target_col=target_col,
    country=country
)

for model_name, results in fitted_sarimax_models.items():
    print("\n" + "=" * 70)
    print(f"{model_name} summary for {country}")
    print("=" * 70)
    print(results.summary())

print(f"\nSARIMAX MODEL COMPARISON FOR {country} ({forecast_year} TEST SET):")
sarimax_comparison_df


SARIMAX Full summary for Norway
                                      SARIMAX Results                                      
Dep. Variable:                     log_load_Norway   No. Observations:                 5472
Model:             SARIMAX(1, 0, 0)x(0, 1, [1], 7)   Log Likelihood               14309.895
Date:                             Sun, 03 May 2026   AIC                         -28597.790
Time:                                     19:00:05   BIC                         -28525.139
Sample:                                 01-08-2010   HQIC                        -28572.440
                                      - 12-31-2024                                         
Covariance Type:                            robust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
HDD_Norway          0.0135      0.000     72.874      0.000    

,Country,Model,Specification,Order,Seasonal Order,RMSE,MAE,Rank
0,Norway,SARIMAX Restricted,Restricted,"(0, 0, 1)","(0, 1, 1, 7)",0.022200,0.017373,1
1,Norway,SARIMAX Full,Full,"(1, 0, 0)","(0, 1, 1, 7)",0.022728,0.018065,2


In [35]:
restricted_results = fitted_sarimax_models["SARIMAX Restricted"]

sensitivity_df = pd.DataFrame({
    "Variable": restricted_results.params.index,
    "Coefficient": restricted_results.params.values,
    "Std_Error": restricted_results.bse.values,
    "p_value": restricted_results.pvalues.values
})

pretty_map = {
    f"HDD_{country}": "HDD",
    f"Extreme_Cold_{code}": "Extreme Cold",
    f"HDD_Extreme_{code}": "HDD × Extreme Cold",
    f"Holiday_{country}": "Holiday",
    f"lag1_log_{lag_prefix}": "Lagged demand"
}

sensitivity_df["Pretty Variable"] = (
    sensitivity_df["Variable"]
    .map(pretty_map)
    .fillna(sensitivity_df["Variable"])
)

output_path = f"../outputs/tables/{country}_restricted_sensitivity_results_{forecast_year}.csv"
sensitivity_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: ../outputs/tables/Norway_restricted_sensitivity_results_2025.csv
